# What Post-Training Changes in a Base Model

Give the same prompts to SmolLM2-360M and its instruct version, read the chat template that separates them, then measure token by token where post-training moved probability.

LLM Training Lab, module 10. Regime: INFERENCE & EVAL. Runs on a CPU. The cells below are the lab page's code, in order.

In [ ]:
# Runtime check. Run this cell first.
import platform, shutil, sys
ok = True
print("Python", platform.python_version())
if sys.version_info < (3, 11):
    ok = False
    print("Python 3.11 or later is needed, and 3.12 is recommended. In Colab use Runtime > Change runtime type, default runtime.")
try:
    import psutil
    ram = psutil.virtual_memory().total / 1e9
    print("RAM", round(ram, 1), "GB")
    if ram < 6:
        ok = False
        print("This lab holds two 360M models in float32, about 3.3 GB. Close other notebooks or use a machine with 8 GB.")
except ImportError:
    pass
free = shutil.disk_usage(".").free / 1e9
print("Free disk", round(free, 1), "GB")
if free < 3:
    ok = False
    print("The two SmolLM2-360M checkpoints need about 1.5 GB of disk. Free some space.")
# Read versions without importing transformers, so the install in the next cell takes effect with no restart.
from importlib.metadata import PackageNotFoundError, version
for pkg in ["torch", "transformers"]:
    try:
        print(pkg, version(pkg))
    except PackageNotFoundError:
        print(pkg, "not installed")
        if pkg == "torch":
            ok = False
            print("FIX: install PyTorch for your machine from pytorch.org. Colab and Kaggle ship it.")
print("No GPU is needed for this lab." if ok else "Fix the items above, then run this cell again.")

In [ ]:
# Pinned Hugging Face versions of the recorded runs. torch and numpy stay as the runtime ships them.
# %pip installs into this notebook's own Python, also on a laptop with a virtual environment.
# pip may print a note about restarting the kernel. Restart only if this cell prints a FIX line.
%pip install -q "transformers==5.22.1" "tokenizers==0.23.3" "huggingface_hub==2.4.0"
import sys
from importlib.metadata import version
print("transformers", version("transformers"), "| tokenizers", version("tokenizers"), "| huggingface_hub", version("huggingface_hub"))
if version("transformers") != "5.22.1":
    print("FIX: the install failed. Read the pip error above, then run this cell again.")
elif "transformers" in sys.modules and sys.modules["transformers"].__version__ != "5.22.1":
    print("FIX: an older transformers is already loaded. Restart the session (Colab: Runtime > Restart session), then run the cells below this one.")
else:
    print("Versions match the recorded runs.")

The next cell writes `prompts.py`, the 40 prompts that steps 5 and 6 use.

In [ ]:
%%writefile prompts.py
"""The fixed prompt set for lltBaseVsInstruct (40 prompts, written for this lab).

Five groups of 8: how-to and advice, explanations, code, writing, maths and reasoning.
The order is fixed, and every Run Library file records which prompts it used by index.
"""

PROMPTS = [
    # how-to and advice
    "How do I make a cup of green tea?",
    "Give me three tips for sleeping better.",
    "How should I prepare for a job interview?",
    "What is a good way to start learning to play the guitar?",
    "How can I keep my houseplants alive?",
    "What should I pack for a weekend hiking trip?",
    "How do I write a polite email asking for a deadline extension?",
    "How can I save money on groceries?",
    # explanations
    "Why is the sky blue?",
    "Explain how a vaccine works.",
    "What causes the seasons on Earth?",
    "What is the difference between weather and climate?",
    "How does a refrigerator keep food cold?",
    "What is photosynthesis?",
    "Why do we have leap years?",
    "What is inflation in economics?",
    # code
    "Write a Python function that reverses a string.",
    "How do I read a CSV file in Python?",
    "What is the difference between a list and a tuple in Python?",
    "Write a Python function that checks whether a number is prime.",
    "Explain what a for loop does.",
    "How do I remove duplicates from a list in Python?",
    "Write a SQL query that counts the rows in a table called orders.",
    "What does the git commit command do?",
    # writing
    "Write a short poem about the ocean.",
    "Write a two-sentence story about a lost dog.",
    "Write a thank-you note to a teacher.",
    "Suggest a name for a small bakery.",
    "Describe a sunset in three sentences.",
    "Write a short product description for a reusable water bottle.",
    "Write a haiku about autumn.",
    "Write an opening line for a mystery novel.",
    # maths and reasoning
    "What is 15 percent of 80?",
    "If a train travels 60 km in 45 minutes, what is its speed in km per hour?",
    "What is the next number in the sequence 2, 4, 8, 16?",
    "A shirt costs 20 dollars after a 20 percent discount. What was the original price?",
    "How many minutes are there in a week?",
    "Is 91 a prime number?",
    "What is the square root of 144?",
    "If I have 3 apples and eat one, how many are left?",
]

assert len(PROMPTS) == 40

## Step 1. Give one prompt to the base and the instruct model

What you should see. The base model answers the sky question with Rayleigh scattering and the maths question with 12. On the tea question it repeats the question until its 40 tokens run out. The instruct model answers the tea question, but on the other two it prints only `<|im_end|>`, its end-of-turn token, and stops.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_num_threads(4)
BASE = "HuggingFaceTB/SmolLM2-360M"
INSTRUCT = "HuggingFaceTB/SmolLM2-360M-Instruct"
tok_b = AutoTokenizer.from_pretrained(BASE)
tok_i = AutoTokenizer.from_pretrained(INSTRUCT)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).eval()
inst = AutoModelForCausalLM.from_pretrained(INSTRUCT, dtype=torch.float32).eval()

@torch.no_grad()
def greedy(model, tok, ids, n):
    # Greedy decoding: always take the most likely next token.
    x = torch.tensor([ids])
    out = model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=n,
                         do_sample=False, pad_token_id=tok.eos_token_id)
    return out[0, len(ids):].tolist()

THREE = ["How do I make a cup of green tea?", "Why is the sky blue?", "What is 15 percent of 80?"]
for prompt in THREE:
    ids = tok_b(prompt)["input_ids"]
    print(prompt)
    for name, tok, model in [("base", tok_b, base), ("instruct", tok_i, inst)]:
        print("  ", name.ljust(9), repr(tok.decode(greedy(model, tok, ids, 40))[:90]))

## Step 2. Read the chat template

What you should see. The template inserts the system prompt **You are a helpful AI assistant named SmolLM, trained by Hugging Face**. The ids are a `BatchEncoding` of 37 tokens, and 33 without the generation prompt. The instruct model's EOS, its end-of-sequence token, is `<|im_end|>` with id 2. The base model's is `<|endoftext|>` with id 0. The base ships no chat template, and both vocabularies match.

In [ ]:
msgs = [{"role": "user", "content": "What is the capital of France?"}]
print(tok_i.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

enc = tok_i.apply_chat_template(msgs, add_generation_prompt=True)
ids = list(enc["input_ids"])
print(type(enc).__name__, list(enc.keys()), len(ids), "tokens")
print(tok_i.convert_ids_to_tokens(ids)[-8:])
no_gen = tok_i.apply_chat_template(msgs, add_generation_prompt=False)
print(len(no_gen["input_ids"]), "tokens without the generation prompt")

print("instruct EOS", repr(tok_i.eos_token), tok_i.eos_token_id, "| base EOS", repr(tok_b.eos_token), tok_b.eos_token_id)
print("base ships a chat template:", tok_b.chat_template is not None)
print("same vocabulary:", tok_b.get_vocab() == tok_i.get_vocab(), len(tok_i), "tokens")

## Step 3. Run both models in the chat format

What you should see. The instruct model now answers all three questions. The base model repeats each question and then fills the rest with `��` and newlines, with the question again, or with fragments of the system prompt and stray words such as **boutique**. The `��` is a byte-level token that is not valid UTF-8 on its own.

In [ ]:
def chat(prompt):
    msgs = [{"role": "user", "content": prompt}]
    return list(tok_i.apply_chat_template(msgs, add_generation_prompt=True)["input_ids"])

for prompt in THREE:
    ids = chat(prompt)
    print(prompt)
    for name, tok, model in [("base", tok_b, base), ("instruct", tok_i, inst)]:
        print("  ", name.ljust(9), repr(tok.decode(greedy(model, tok, ids, 40))[:90]))

## Step 4. Score one response with per-token KL

What you should see. The answer opens with **The sky is blue because of a phenomenon known as Rayleigh scattering**. The first token, **The**, has KL 2.575 and base rank 6, so it is shifted. **sky** has KL 2.238 at rank 1. **Ray** and **leigh** are both below 0.07. Over all 82 tokens, the end token included, the mean KL is 0.651 nats. The answer's last full sentence is wrong, since red and orange light scatter much less than blue, not in a more random manner. KL measures how far post-training shifted the predictions, not whether the answer is correct.

In [ ]:
import numpy as np

@torch.no_grad()
def logprobs(model, ctx, resp):
    # Row t holds log-probabilities, the natural log of each probability, for response token t over the whole vocabulary.
    x = torch.tensor([ctx + resp])
    logits = model(input_ids=x).logits[0, len(ctx) - 1 : len(ctx) - 1 + len(resp)]
    return torch.log_softmax(logits.double(), dim=-1)

def kl_and_rank(lp_i, lp_b, resp):
    kl = (lp_i.exp() * (lp_i - lp_b)).sum(-1)          # KL(instruct || base) per token, in nats
    chosen = lp_b[torch.arange(len(resp)), torch.tensor(resp)]
    rank = (lp_b > chosen[:, None]).sum(-1) + 1         # 1 = the base model also put this token first
    return kl, rank

ctx = chat("Why is the sky blue?")
resp = greedy(inst, tok_i, ctx, 128)
kl, rank = kl_and_rank(logprobs(inst, ctx, resp), logprobs(base, ctx, resp), resp)
for t in range(12):
    print(str(t + 1).rjust(3), repr(tok_i.decode([resp[t]])).ljust(16), "KL", round(float(kl[t]), 3), "base rank", int(rank[t]))
print(len(resp), "tokens, mean KL", round(float(kl.mean()), 3), "nats")

## Step 5. Measure where the KL sits across 40 prompts

What you should see. 4,048 response tokens. Your count can differ by a few tokens on other hardware. Unshifted 0.74, marginal 0.163 and shifted 0.097, so 90.3% of the instruct model's tokens sit in the base model's top 3. The position lines match the table. The most frequent shifted tokens include answer openers such as **The**, a blank-line token, **This**, **A**, **Here** and **To**, and also **the**, **a**, **and** and a comma inside sentences.

In [ ]:
from collections import Counter
from prompts import PROMPTS          # 40 prompts in five groups of 8, written by the notebook's first cell

# The long cell of the lab: 40 greedy answers, each scored by both models.
USE_OUR_ANSWERS = False   # True: load our 40 answers and their per-token scores from the Run Library instead
OUR_RUN = "https://raw.githubusercontent.com/genieincodebottle/aiml-companion/main/courses/llm-training/runs/lltBaseVsInstruct/lltBaseVsInstruct-t1-smollm2-kl-v2026-11.json"
ours = {}
if USE_OUR_ANSWERS:
    import json, urllib.request
    try:
        ours = {s["prompt"]: s for s in json.load(urllib.request.urlopen(OUR_RUN, timeout=60))["samples"]}
    except OSError as err:
        raise RuntimeError("Our run file is not reachable (" + str(err) + "). Set USE_OUR_ANSWERS = False to compute the answers.") from err

rows, responses = [], {}
for p in PROMPTS:
    if p in ours:                                    # our answer, with our KL and base rank for each token
        resp = ours[p]["responseIds"]
        kl, rank = [x[1] for x in ours[p]["perToken"]], [x[2] for x in ours[p]["perToken"]]
    else:
        ctx = chat(p)
        resp = greedy(inst, tok_i, ctx, 128)
        kl, rank = kl_and_rank(logprobs(inst, ctx, resp), logprobs(base, ctx, resp), resp)
    responses[p] = resp
    for t, r in enumerate(resp):
        if r != tok_i.eos_token_id:                  # the end token is scored on its own below
            rows.append((t + 1, float(kl[t]), int(rank[t]), tok_i.decode([r]).strip() or repr(tok_i.decode([r]))))

pos = np.array([r[0] for r in rows]); kl_all = np.array([r[1] for r in rows]); rk = np.array([r[2] for r in rows])
print(len(rows), "response tokens from", len(PROMPTS), "prompts")
print("unshifted, base rank 1  ", round(float((rk == 1).mean()), 3))
print("marginal, base rank 2-3 ", round(float(((rk > 1) & (rk <= 3)).mean()), 3))
print("shifted, base rank > 3  ", round(float((rk > 3).mean()), 3))
for lo, hi in [(1, 1), (2, 5), (6, 20), (21, 60), (61, 128)]:
    m = (pos >= lo) & (pos <= hi)
    print("positions", lo, "to", hi, "|", round(float(m.mean()), 3), "of tokens |",
          round(float(kl_all[m].sum() / kl_all.sum()), 3), "of the KL")
print(Counter(r[3] for r in rows if r[2] > 3).most_common(12))

## Step 6. Give the base model its own question format

What you should see. Chat tokens give mean KL 0.525, unshifted 0.74, shifted 0.097 and 0.165 of the KL in the first 5 tokens. The plain question gives 0.43, 0.761, 0.077 and 0.122. Both lines still put the largest per-token KL at the first token.

In [ ]:
PLAIN = """Question: {p}
Answer:
"""

rows_plain = []
for p in PROMPTS:
    resp = responses[p]
    ctx_i, ctx_b = chat(p), tok_b(PLAIN.format(p=p))["input_ids"]
    kl, rank = kl_and_rank(logprobs(inst, ctx_i, resp), logprobs(base, ctx_b, resp), resp)
    for t, r in enumerate(resp):
        if r != tok_i.eos_token_id:
            rows_plain.append((t + 1, float(kl[t]), int(rank[t])))

pos_p = np.array([r[0] for r in rows_plain]); kl_p = np.array([r[1] for r in rows_plain]); rk_p = np.array([r[2] for r in rows_plain])
for name, k, r, ps in [("chat tokens", kl_all, rk, pos), ("plain question", kl_p, rk_p, pos_p)]:
    print(name.ljust(15), "mean KL", round(float(k.mean()), 3), "| unshifted", round(float((r == 1).mean()), 3),
          "| shifted", round(float((r > 3).mean()), 3), "| KL in first 5 tokens", round(float(k[ps <= 5].sum() / k.sum()), 3))

## Step 7. Map the stages between the two checkpoints

What you should see. You can name each stage's starting weights, what changed and its objective. In this lab you changed nothing, so its regime is INFERENCE & EVAL.

## Prove it 1

Write `kl_per_row`, `base_rank` and `shift_label` in NumPy, the three functions behind steps 4 and 5. Run the cell, and it prints All checks passed when all three are right. The checks include a zero probability, which must add 0 to the KL, and a tie for first place.

In [ ]:
import numpy as np

def kl_per_row(p, q):
    # p, q: arrays of shape (T, V), each row a probability distribution.
    # Return KL(p || q) for each row, in nats. A zero in p adds nothing (0 log 0 = 0).
    raise NotImplementedError("write kl_per_row")

def base_rank(q, tokens):
    # q: (T, V) base-model distributions. tokens: the token id the instruct model chose at each row.
    # Rank 1 means the base model also put that token first. Ties count in the token's favour.
    raise NotImplementedError("write base_rank")

def shift_label(rank):
    # URIAL's three groups: rank 1 unshifted, rank 2 or 3 marginal, above 3 shifted.
    raise NotImplementedError("write shift_label")

# Checks. Run the cell; it prints All checks passed when all three functions are right.
p = np.array([[0.90, 0.05, 0.03, 0.02, 0.00],
              [0.02, 0.92, 0.03, 0.02, 0.01],
              [0.01, 0.02, 0.95, 0.01, 0.01],
              [0.01, 0.01, 0.02, 0.06, 0.90]])
q = np.array([[0.10, 0.60, 0.20, 0.05, 0.05],
              [0.05, 0.85, 0.05, 0.03, 0.02],
              [0.02, 0.03, 0.90, 0.03, 0.02],
              [0.15, 0.30, 0.30, 0.20, 0.05]])
kl = kl_per_row(p, q)
assert kl.shape == (4,), "return one KL value per row"
assert np.all(np.isfinite(kl)), "a zero in p must add 0, not nan"
assert np.allclose(kl, [1.7780, 0.0241, 0.0184, 2.4138], atol=1e-4), np.round(kl, 4)
assert np.allclose(kl_per_row(q, q), 0.0), "KL of a distribution with itself is 0"
assert kl_per_row(q, p[[1, 1, 2, 3]]).min() >= 0, "KL is never negative"
tokens = p.argmax(axis=1)
rank = base_rank(q, tokens)
assert rank.tolist() == [3, 1, 1, 5], rank.tolist()
assert base_rank(np.array([[0.5, 0.5]]), np.array([1])).tolist() == [1], "a tie with the top token is rank 1"
assert shift_label(rank).tolist() == ["marginal", "unshifted", "unshifted", "shifted"], shift_label(rank).tolist()
share = kl / kl.sum()
assert round(float(share[[0, 3]].sum()), 3) == 0.990, round(float(share[[0, 3]].sum()), 3)
print("KL per row", np.round(kl, 4), "| base ranks", rank.tolist())
print("Rows 1 and 4 hold", f"{share[[0, 3]].sum():.1%}", "of the total KL")
print("All checks passed")